In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)


In [ ]:
import subprocess
import sys
import os
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob
from rdkit import Chem
import signal
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta
from rdkit.Chem import rdMolDescriptors
import numpy as np

In [ ]:
df_reaction = pd.read_excel('HTQC_global_reaction_descriptors.xlsx')
df_reaction

In [ ]:
fchk_path_neutral = "energy/neutral/success"
fchk_path_oxidation = "energy/oxidation/success"
fchk_path_reduction = "energy/reduction/success"
gjf_path_base = "energy/neutral/success"

In [ ]:

try:
    from openpyxl import load_workbook
except ImportError:
    print("Public message removed for release.")
    exit()

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def extract_energy_from_out(file_path):
    energy = None
    with open(file_path, 'r') as file:
        lines = file.readlines()
        energy_line = ''
        for i, line in enumerate(lines):
            
            line = line.strip().replace('\n', '')
            
            if line.strip().endswith('H') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('F='):
                    
                    line = line.strip() + next_line.strip()
            
            if line.strip().endswith('HF') and i+1 < len(lines):
                next_line = lines[i+1]
                if next_line.strip().startswith('='):
                    
                    line = line.strip() + next_line.strip()
            
            if 'HF=' in line:
                
                start = line.find('HF=') + 3  
                energy_line = line[start:]
                
                if '\\' in energy_line:
                    energy = energy_line.split('\\')[0].strip()
                    return energy
                else:
                    
                    for j in range(i+1, len(lines)):
                        energy_line += lines[j].strip()
                        if '\\' in lines[j]:
                            energy = energy_line.split('\\')[0].strip()
                            return energy
    return energy  

In [ ]:
def extract_dipole_moment(file_path):
    
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()  

        
        dipole_moment_lines = [
            i for i, line in enumerate(lines)
            if re.search(r"Dipole moment \(field-independent basis, Debye\):", line)
        ]

        
        if not dipole_moment_lines:
            print("Public message removed for release.")
            return None

        
        last_line_number = dipole_moment_lines[-1]

        
        next_line = lines[last_line_number + 1]
        match = re.search(r"Tot=\s*(-?\d+\.\d+)", next_line)

        
        if match:
            return float(match.group(1))
        else:
            print("Public message removed for release.")
            return None

    except FileNotFoundError:
        print("Public message removed for release.")
        return None
    except Exception as e:
        print("Public message removed for release.")
        return None

In [ ]:

def extract_homo_lumo(file_path):
    
    try:
        filename = get_filename_without_extension(file_path)
        
        with open(file_path, 'r') as file:
            content = file.readlines()
        
        
        pop_analysis_lines = [i for i, line in enumerate(content) if re.match(r'\s*Population analysis', line)]
        
        
        if len(pop_analysis_lines) == 0:
            raise ValueError("Public message removed for release.")

        
        last_population_analysis = pop_analysis_lines[-1]

        
        homo, lumo = None, None
        for line_number in range(last_population_analysis, len(content)):
            occ_match = re.match(r'\s*Alpha\s+occ.\s+eigenvalues\s+--(.*)', content[line_number])
            virt_match = re.match(r'\s*Alpha\s+virt.\s+eigenvalues\s+--(.*)', content[line_number + 1]) if line_number + 1 < len(content) else None
            
            if occ_match and virt_match:
                
                homo = float(occ_match.group(1).strip().split()[-1])  
                lumo = float(virt_match.group(1).strip().split()[0])  
                break
        
        
        if homo is None or lumo is None:
            raise ValueError("Public message removed for release.")
        
        
        return homo, lumo

    except FileNotFoundError:
        
        raise FileNotFoundError("Public message removed for release.")
    except Exception as e:
        
        raise Exception("Public message removed for release.")

In [ ]:

def extract_entropy(file_path):
    
    entropy_pattern = re.compile(r'\s*Total\s+[0-9.-]+\s+[0-9.-]+\s+([0-9.-]+)')
    try:
        with open(file_path, 'r') as file:
            file_content = file.read()  
            match = entropy_pattern.search(file_content)
            if match:
                
                entropy_value_cal = float(match.group(1))
                
                entropy_value_joules = entropy_value_cal * 4.184 
                return entropy_value_joules
    except FileNotFoundError:
        print(f"The file {file_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None  

In [ ]:

def extract_enthalpy_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Enthalpy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:
def extract_properties_from_energycal(excel_file, neutral_success_dir, oxidation_success_dir, reduction_success_dir):

    
    df = pd.read_excel(excel_file)
        
    
    if 'Neutral Energy (Hatree)' not in df.columns:
        df['Neutral Energy (Hatree)'] = 0.0
    if 'Oxidation Energy (Hatree)' not in df.columns:
        df['Oxidation Energy (Hatree)'] = 0.0
    if 'Reduction Energy (Hatree)' not in df.columns:
        df['Reduction Energy (Hatree)'] = 0.0
    
    
    dirs_and_columns = {
        neutral_success_dir: 'Neutral Energy (Hatree)',
        oxidation_success_dir: 'Oxidation Energy (Hatree)',
        reduction_success_dir: 'Reduction Energy (Hatree)'
    }
    
    
    for success_dir, column_name in dirs_and_columns.items():
        
        if not os.path.exists(success_dir):
            print(f"Directory {success_dir} does not exist.")
            continue

        
        for filename in os.listdir(success_dir):
            if filename.endswith('.out'):
                
                component_name = os.path.splitext(filename)[0]
                
                energy = extract_energy_from_out(os.path.join(success_dir, filename))
                
                
                energy = float(energy) if energy else 0.0
                
                
                df.loc[df['Name'] == component_name, column_name] = energy

    
    df.to_excel(excel_file, index=False)

In [ ]:

def extract_gibbs_correction(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            if "Thermal correction to Gibbs Free Energy" in line:
                
                match = re.search(r"=\s*([-\d.]+)", line)
                if match:
                    return float(match.group(1))
    return None

In [ ]:

def extract_properties_from_opt_freq(excel_file, success_dir):
    
    
    df = pd.read_excel(excel_file)
    
    
    
    df["Thermal correction to Gibbs Free Energy (Hatree)"] = 0.0 
    

    df["Thermal correction to Enthalpy (Hatree)"] = 0.0 
    

    df['Entropy (J/mol·K)'] = 0.0 
    
    df['HOMO (Hatree)'] = 0.0 
    df['LUMO (Hatree)'] = 0.0 
    
    df['Dipole (Debye)'] = 0.0 
    
    
    for filename in os.listdir(success_dir):
        if filename.endswith('.out'):
            
            component_name = os.path.splitext(filename)[0]
            
            gibbs_correction = extract_gibbs_correction(os.path.join(success_dir, filename)) 
            enthalpy_correction = extract_enthalpy_correction(os.path.join(success_dir, filename)) 
            entropy = extract_entropy(os.path.join(success_dir, filename)) 
            homo, lumo = extract_homo_lumo(os.path.join(success_dir, filename)) 
            dipole = extract_dipole_moment(os.path.join(success_dir, filename))
            
            
            dipole = float(dipole) if dipole else 0.0
            
            
            
            
            df.loc[df['Name'] == component_name, 'Thermal correction to Gibbs Free Energy (Hatree)'] = gibbs_correction
            df.loc[df['Name'] == component_name, 'Thermal correction to Enthalpy (Hatree)'] = enthalpy_correction
            df.loc[df['Name'] == component_name, 'Entropy (J/mol·K)'] = entropy
            df.loc[df['Name'] == component_name, 'HOMO (Hatree)'] = homo
            df.loc[df['Name'] == component_name, 'LUMO (Hatree)'] = lumo
            df.loc[df['Name'] == component_name, 'Dipole (Debye)'] = dipole
                   
    
    df.to_excel(excel_file, index=False)

In [ ]:


def calculate_global_reactivity_descriptors(mol_excel_path):
    
    
    
    df = pd.read_excel(mol_excel_path)
    
    
    df['IP'] = df['Oxidation Energy (Hatree)'] - df['Neutral Energy (Hatree)']
    df['EA'] = df['Neutral Energy (Hatree)'] - df['Reduction Energy (Hatree)']
    
    
    df['Mulliken Electronegativity'] = (df['IP'] + df['EA']) / 2
    
    
    df['Chemical Potential'] = -df['Mulliken Electronegativity']
    
    
    df['Hardness'] = df['IP'] - df['EA']
    
    
    df['Softness'] = 1 / df['Hardness']
    
    
    df['Electrophilicity Index'] = (df['Chemical Potential'] ** 2) / (2 * df['Hardness'])
    
    
    HOMO_TCE = -0.34418  
    df['Nucleophilicity Index'] = df['HOMO (Hatree)'] - HOMO_TCE
    
    
    df.to_excel(mol_excel_path, index=False)
    
    print("Public message removed for release.")

In [ ]:

def data_processing(excel_file):
    
    df = pd.read_excel(excel_file)
       
    df["Gibbs Free Energy (Hatree)"] = df['Neutral Energy (Hatree)'] + df["Thermal correction to Gibbs Free Energy (Hatree)"] 
    df["Enthalpy (Hatree)"] = df['Neutral Energy (Hatree)'] + df["Thermal correction to Enthalpy (Hatree)"] 
    df["HOMO LUMO Gap (eV)"] = (df['LUMO (Hatree)'] - df["HOMO (Hatree)"]) * 27.2 

    
    
    
    
    
    df.to_excel(excel_file, index=False)

In [ ]:

def convert_chk_to_fchk(Gaussian_path):
    
    for filename in os.listdir(Gaussian_path):
        
        if filename.endswith('.chk'):
            
            base_filename = os.path.splitext(filename)[0]
            
            input_file = os.path.join(Gaussian_path, filename)
            
            output_file = os.path.join(Gaussian_path, base_filename + '.fchk')
            
            command = ['formchk', input_file, output_file]
            
            try:
                
                subprocess.run(command, check=True)
                print(f"Converted {input_file} to {output_file}")
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {input_file}: {e}")

In [ ]:

convert_chk_to_fchk(fchk_path_neutral)
convert_chk_to_fchk(fchk_path_oxidation)
convert_chk_to_fchk(fchk_path_reduction)

In [ ]:



def calculate_molecule_Vsmin(fchk_path):
    
    commands = '12\n0\n11\nn\n-1\n-1\nq\n'  
    
    
    process = subprocess.Popen(
        ['Multiwfn', fchk_path],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,  
        stderr=subprocess.PIPE,
        text=True
    )
        
    
    stdout, stderr = process.communicate(commands)  
    output = stdout 
    
    
    if process.returncode != 0:
        raise Exception("Public message removed for release.")
    
    
    match_vsmin = re.search(r'Minimal value:\s*(-?\d+\.\d+)\s*kcal/mol', output)
    if match_vsmin:
        Vs_min = float(match_vsmin.group(1))  
    else:
        raise ValueError("Public message removed for release.")
    
    
    
    atom_data_pattern = re.compile(r'\s*(\d+)\s+([\d\.]+)\s+([\d\.]+)\s+([\d\.]+)\s+([-?\d\.]+)\s+([-?\d\.]+)')
    
    
    atom_data = []
    for line in output.splitlines():
        match = atom_data_pattern.match(line)
        if match:
            atom_data.append([int(match.group(1)), 
                              float(match.group(2)), 
                              float(match.group(3)), 
                              float(match.group(4)), 
                              float(match.group(5)), 
                              float(match.group(6))])

    
    df_Electrostatic_Potential = pd.DataFrame(atom_data, columns=['Atom#', 'All area', 'Positive area', 'Negative area', 'Minimal value', 'Maximal value'])

    
    min_value_row = df_Electrostatic_Potential.loc[df_Electrostatic_Potential['Minimal value'].idxmin()]
    
    pass
    
    
    atom_with_Vs_min = min_value_row['Atom#']
    
    pass

    
    
    
    
    return Vs_min, int(atom_with_Vs_min), df_Electrostatic_Potential

In [ ]:


def extract_atom_name(gjf_path, atom_with_Vs_min):
    
    
    
    atom_with_Vs_min = int(atom_with_Vs_min)
    
    
    with open(gjf_path, 'r') as file:
        lines = file.readlines()
    
    
    atom_pattern = re.compile(r'([A-Za-z]+)\s+([-?\d\.]+)\s+([-?\d\.]+)\s+([-?\d\.]+)')
    
    
    atom_list = []
    
    
    for line in lines:
        match = atom_pattern.match(line.strip())  
        if match:
            atom_list.append(match.group(1))  
    
    
    if 0 < atom_with_Vs_min <= len(atom_list):
        atom_name_with_Vs_min = atom_list[atom_with_Vs_min - 1]  
        return atom_name_with_Vs_min
    else:
        raise ValueError("Public message removed for release.")

In [ ]:



def calculate_hirshfeld_charges(fchk_path, category_choice="N"):
    
    
    commands = '7\n1\n1\ny\n0\nq\n'
    
    
    process = subprocess.Popen(
        ['Multiwfn', fchk_path],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    
    stdout, stderr = process.communicate(commands)
    output = stdout  
    
    
    if process.returncode != 0:
        raise Exception("Public message removed for release.")
    
    
    base_filename = os.path.splitext(os.path.basename(fchk_path))[0]
    
    
    chg_file_path = base_filename + '.chg'
    
    try:
        with open(chg_file_path, 'r') as file:
            chg_lines = file.readlines()
    except FileNotFoundError:
        raise FileNotFoundError("Public message removed for release.")
    
    
    atom_names = []
    charges = []
    
    
    for line in chg_lines:
        
        columns = line.strip().split()
        if len(columns) < 5:
            continue  
        atom_name = columns[0]
        charge = float(columns[-1])  
        
        atom_names.append(atom_name)
        charges.append(charge)
    
    
    charge_column_name = f"Hirshfeld charge ({category_choice})"
    
    
    df_charges = pd.DataFrame({
        'Atom name': atom_names,
        charge_column_name: charges
    })
    
    
    return df_charges

In [ ]:

def calculate_simplified_fukui_functions(df_charges):
    
    
    df_charges['f+'] = df_charges['Hirshfeld charge (N)'] - df_charges['Hirshfeld charge (N+1)']
    
    
    df_charges['f-'] = df_charges['Hirshfeld charge (N-1)'] - df_charges['Hirshfeld charge (N)']
    
    
    df_charges['f0'] = (df_charges['Hirshfeld charge (N-1)'] - df_charges['Hirshfeld charge (N+1)']) / 2
    
    
    df_charges['delta_f'] = 2 * df_charges['Hirshfeld charge (N)'] - df_charges['Hirshfeld charge (N+1)'] - df_charges['Hirshfeld charge (N-1)']
    
    return df_charges

In [ ]:
def process_molecules(df_reaction, fchk_path_neutral, fchk_path_oxidation, fchk_path_reduction, gjf_path_base):

    
    for index, row in df_reaction.iterrows():
        name = row['Name']
        
        mol_fchk_path_neutral = os.path.join(fchk_path_neutral, f"{name}.fchk")
        mol_fchk_path_oxidation = os.path.join(fchk_path_oxidation, f"{name}.fchk")
        mol_fchk_path_reduction = os.path.join(fchk_path_reduction, f"{name}.fchk")
        gjf_path = os.path.join(gjf_path_base, f"{name}.gjf")

        
        if not os.path.exists(mol_fchk_path_neutral):
            print("Public message removed for release.")
            continue
        if not os.path.exists(mol_fchk_path_oxidation):
            print("Public message removed for release.")
            continue
        if not os.path.exists(mol_fchk_path_reduction):
            print("Public message removed for release.")
            continue
        
        
        
        try:
            df_charges_neutral = calculate_hirshfeld_charges(mol_fchk_path_neutral, category_choice='N')
        except Exception as e:
            print("Public message removed for release.")
            continue

        
        try:
            df_charges_oxidation = calculate_hirshfeld_charges(mol_fchk_path_oxidation, category_choice='N-1')
        except Exception as e:
            print("Public message removed for release.")
            continue

        
        try:
            df_charges_reduction = calculate_hirshfeld_charges(mol_fchk_path_reduction, category_choice='N+1')
        except Exception as e:
            print("Public message removed for release.")
            continue

        
        chg_files = glob.glob('*.chg')
        
        for file in chg_files:
            try:
                os.remove(file)
                print("Public message removed for release.")
            except Exception as e:
                print("Public message removed for release.")
        
        
        df_charges_neutral.reset_index(drop=True, inplace=True)
        df_charges_oxidation.reset_index(drop=True, inplace=True)
        df_charges_reduction.reset_index(drop=True, inplace=True)
        
        
        df_charges_neutral['Atom Index'] = df_charges_neutral.index
        df_charges_oxidation['Atom Index'] = df_charges_oxidation.index
        df_charges_reduction['Atom Index'] = df_charges_reduction.index
        
        
        df_charges = df_charges_neutral.copy()
        
        
        df_charges = df_charges.merge(
            df_charges_oxidation[['Atom Index', 'Hirshfeld charge (N-1)']],
            on='Atom Index',
            how='left'
        )
        
        
        df_charges = df_charges.merge(
            df_charges_reduction[['Atom Index', 'Hirshfeld charge (N+1)']],
            on='Atom Index',
            how='left'
        )
        
        df_charges.drop(columns=['Atom Index'], inplace=True)
        
        
        df_charges = calculate_simplified_fukui_functions(df_charges)
        
        
        
        output_filename = f"{name}_fukui.csv"
        df_charges.to_csv(output_filename, index=False)
        print("Public message removed for release.")
        

process_molecules(df_reaction, fchk_path_neutral, fchk_path_oxidation, fchk_path_reduction, gjf_path_base)

In [ ]:

def move_fukui_files():
    
    target_folder = 'fukui'
    
    
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print("Public message removed for release.")
    
    
    fukui_files = glob.glob('*_fukui.csv')
    
    
    for file in fukui_files:
        try:
            shutil.move(file, target_folder)
            print("Public message removed for release.")
        except Exception as e:
            print("Public message removed for release.")
    
    if not fukui_files:
        print("Public message removed for release.")


move_fukui_files()

In [ ]:

mol_excel_path = 'HTQC_global_reaction_descriptors.xlsx'
opt_success_dir = "opt+freq/success"
neutral_success_dir = "energy/neutral/success"
oxidation_success_dir = "energy/oxidation/success"
reduction_success_dir = "energy/reduction/success"


extract_properties_from_energycal(mol_excel_path, neutral_success_dir, oxidation_success_dir, reduction_success_dir)
extract_properties_from_opt_freq(mol_excel_path, opt_success_dir)
data_processing(mol_excel_path)

calculate_global_reactivity_descriptors(mol_excel_path)

In [ ]:
def update_mol_fukui_csv(mol_fukui_csv_path, mol_excel_path):
    

    
    mol_fukui_csv_df = pd.read_csv(mol_fukui_csv_path)

    
    name_with_ext = os.path.basename(mol_fukui_csv_path)
    name = os.path.splitext(name_with_ext)[0]

    
    mol_name = name.replace('_fukui', '')

    
    mol_excel_df = pd.read_excel(mol_excel_path)

    
    mol_row = mol_excel_df.loc[mol_excel_df['Name'] == mol_name]

    if mol_row.empty:
        print("Public message removed for release.")
        return

    Electrophilicity_Index = mol_row['Electrophilicity Index'].values[0]
    Nucleophilicity_Index = mol_row['Nucleophilicity Index'].values[0]
    Softness = mol_row['Softness'].values[0]

    
    mol_fukui_csv_df['Local Electrophilicity Index'] = Electrophilicity_Index * mol_fukui_csv_df['f+']
    mol_fukui_csv_df['Local Nucleophilicity Index'] = Nucleophilicity_Index * mol_fukui_csv_df['f-']
    mol_fukui_csv_df['Local Softness+'] = Softness * mol_fukui_csv_df['f+']
    mol_fukui_csv_df['Local Softness-'] = Softness * mol_fukui_csv_df['f-']
    mol_fukui_csv_df['Local Softness0'] = Softness * mol_fukui_csv_df['f0']

    
    with np.errstate(divide='ignore', invalid='ignore'):
        mol_fukui_csv_df['Relative Electrophilicity Index'] = mol_fukui_csv_df['Local Softness+'] / mol_fukui_csv_df['Local Softness-']
        mol_fukui_csv_df['Relative Nucleophilicity Index'] = mol_fukui_csv_df['Local Softness-'] / mol_fukui_csv_df['Local Softness+']

    
    mol_fukui_csv_df['Relative Electrophilicity Index'].replace([np.inf, -np.inf], np.nan, inplace=True)
    mol_fukui_csv_df['Relative Electrophilicity Index'].fillna(0, inplace=True)

    mol_fukui_csv_df['Relative Nucleophilicity Index'].replace([np.inf, -np.inf], np.nan, inplace=True)
    mol_fukui_csv_df['Relative Nucleophilicity Index'].fillna(0, inplace=True)

    
    mol_fukui_csv_df.to_csv(mol_fukui_csv_path, index=False)
    print("Public message removed for release.")

In [ ]:




fukui_folder = 'fukui'

for filename in os.listdir(fukui_folder):
    if filename.endswith('_fukui.csv'):
        mol_fukui_csv_path = os.path.join(fukui_folder, filename)
        update_mol_fukui_csv(mol_fukui_csv_path, mol_excel_path)